
# Hybrid Movie Recommendation System

## 1. Introduction

The rapid growth of digital movie platforms has created a need for recommendation systems that help users discover movies based on their interests and preferences.

Traditional recommendation approaches have certain limitations. Content-based filtering recommends movies using their attributes, such as genres and tags, but may struggle to capture the preferences of users with similar interests. Collaborative filtering uses interactions and ratings from multiple users, but can suffer from data sparsity and cold-start problems.

This project develops a Hybrid Movie Recommendation System that combines content-based filtering and collaborative filtering to generate personalized movie recommendations.

The system uses the MovieLens 32M dataset containing movie information, user ratings and user-generated tags.

## 1.1 Objectives

- Explore and preprocess the MovieLens 32M dataset.
- Develop a content-based recommendation model using TF-IDF and cosine similarity.
- Develop an item-based collaborative filtering model using nearest neighbours.
- Combine both approaches using weighted Reciprocal Rank Fusion.
- Evaluate the recommendation models using Precision@10, Recall@10 and NDCG@10.
- Experiment with different hybrid weights to identify a suitable configuration.

## 1.2 Proposed Approach

The proposed system combines two recommendation techniques:

1. **Content-Based Filtering:** Uses movie genres and tags to identify movies with similar characteristics.
2. **Collaborative Filtering:** Uses patterns in user ratings to identify movies preferred by users with similar movie interests.

The outputs of both recommendation models are combined using weighted Reciprocal Rank Fusion. The final experimental configuration assigns 20% weight to content-based recommendations and 80% weight to collaborative recommendations.



## 2. Dataset Description

This project uses the **MovieLens 32M dataset**, a movie-rating dataset containing user ratings, movie metadata and user-generated tags.

The dataset consists of four main files:

| File | Description |
|---|---|
| `movies.csv` | Movie IDs, titles and genres |
| `ratings.csv` | User IDs, movie IDs, ratings and timestamps |
| `tags.csv` | User-generated tags associated with movies |
| `links.csv` | Links between MovieLens movie IDs and external movie databases |

### 2.1 Dataset Statistics

The dataset contains:

- **87,585** unique movies
- **32,000,204** user ratings
- **2,000,072** tag records
- **200,948** unique users

The ratings range from 0.5 to 5.0, with an average rating of approximately 3.54.

### 2.2 Dataset Characteristics

The dataset is highly sparse because each user rates only a small fraction of the available movies.

The user-item rating matrix has approximately **99.82% sparsity**. Therefore, constructing a dense rating matrix would require a large amount of memory.

To address this, the collaborative filtering component uses sparse matrix representations.

The dataset also contains missing values in some tag and external movie ID fields. These are handled during preprocessing.



## 3. Exploratory Data Analysis

Exploratory Data Analysis was performed to understand the structure, distribution and characteristics of the MovieLens 32M dataset before building the recommendation models.

### 3.1 Rating Analysis

The ratings dataset contains 32,000,204 records from 200,948 unique users.

The rating statistics are:

| Statistic | Value |
|---|---:|
| Mean rating | 3.5404 |
| Standard deviation | 1.0590 |
| Minimum rating | 0.5 |
| Maximum rating | 5.0 |
| Median rating | 3.5 |

The average rating of approximately 3.54 indicates that users tend to give ratings slightly above the midpoint of the rating scale.

### 3.2 User Activity Analysis

The number of ratings contributed by users varies considerably.

- Total unique users: 200,948
- Average ratings per user: 159.25
- Median ratings per user: 73
- Maximum ratings by a single user: 33,332

This variation indicates that some users contribute substantially more rating information than others.

### 3.3 Missing Values and Data Quality

The dataset was inspected for missing values and duplicate records.

- The tags data contains 17 missing tag values.
- The links data contains 124 missing TMDB IDs.
- No duplicate rows were identified in the inspected dataset files.

Missing tag values are excluded during tag preprocessing. Missing external movie IDs do not prevent the recommendation models from using the movie ratings and metadata.

### 3.4 User-Item Matrix Sparsity

A user-item matrix represents users as rows and movies as columns, with ratings stored at the corresponding positions.

The matrix is approximately **99.82% sparse**, meaning that most users have not rated most movies.

A dense representation would require approximately 65.6 GiB of memory using 32-bit floating-point values. Therefore, sparse matrix representations are used in the collaborative filtering model to reduce memory consumption.

### 3.5 EDA Findings

The analysis highlights three important characteristics:

1. The dataset contains a large number of ratings and users.
2. User activity is uneven, with substantial differences in the number of ratings contributed by users.
3. The user-item matrix is highly sparse, making sparse data structures important for collaborative filtering.


In [21]:
import pandas as pd
import numpy as np

data_folder = "data/ml-32m"

# Load the four MovieLens datasets
movies = pd.read_csv("data/ml-32m/movies.csv")
ratings = pd.read_csv("data/ml-32m/ratings.csv")
tags = pd.read_csv("data/ml-32m/tags.csv")
links = pd.read_csv("data/ml-32m/links.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [22]:

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Links:", links.shape)

Movies: (87585, 3)
Ratings: (32000204, 4)
Tags: (2000072, 4)
Links: (87585, 3)


In [23]:

print("MOVIES")
display(movies.head())

print("RATINGS")
display(ratings.head())

print("TAGS")
display(tags.head())

print("LINKS")
display(links.head())


MOVIES


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


RATINGS


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


TAGS


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


LINKS


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [24]:

datasets = {
    "Movies": movies,
    "Ratings": ratings,
    "Tags": tags,
    "Links": links
}

for name, dataset in datasets.items():
    print(name)

    print("Data types:")
    print(dataset.dtypes)

    print("\nMissing values:")
    print(dataset.isnull().sum())

    print("\nDuplicate rows:", dataset.duplicated().sum())


Movies
Data types:
movieId     int64
title      object
genres     object
dtype: object

Missing values:
movieId    0
title      0
genres     0
dtype: int64

Duplicate rows: 0
Ratings
Data types:
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate rows: 0
Tags
Data types:
userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object

Missing values:
userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

Duplicate rows: 0
Links
Data types:
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object

Missing values:
movieId      0
imdbId       0
tmdbId     124
dtype: int64

Duplicate rows: 0


In [25]:

# Basic statistics of movie ratings
print("Rating statistics:")
display(ratings["rating"].describe())

# Count of each rating
rating_distribution = ratings["rating"].value_counts().sort_index()

print("\nRating distribution:")
display(rating_distribution)

# Percentage distribution
rating_percentages = (
    ratings["rating"].value_counts(normalize=True)
    .sort_index() * 100
)

print("\nRating percentages:")
display(rating_percentages)


Rating statistics:


count    3.200020e+07
mean     3.540396e+00
std      1.058986e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


Rating distribution:


rating
0.5     525132
1.0     946675
1.5     531063
2.0    2028622
2.5    1685386
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
Name: count, dtype: int64


Rating percentages:


rating
0.5     1.641027
1.0     2.958341
1.5     1.659561
2.0     6.339403
2.5     5.266798
3.0    18.921723
3.5    13.406493
4.0    26.148752
4.5     9.293691
5.0    14.364212
Name: proportion, dtype: float64

In [26]:

# Extract release year from the movie title
movies["year"] = movies["title"].str.extract(
    r"\((\d{4})\)$"
).astype("Int64")

# Find the latest release year
latest_year = movies["year"].max()

# Display movies from the latest year
latest_movies = movies[
    movies["year"] == latest_year
][["title", "genres"]]

print("Latest release year:", latest_year)
display(latest_movies)


Latest release year: 2023


,title,genres
84230,Asterix & Obelix: The Middle Kingdom (2023),Adventure|Children|Comedy
84254,The Blue Caftan (2023),Drama|Romance
84330,Bad Behaviour (2023),Comedy|Drama
84331,Cassandro (2023),(no genres listed)
84333,Eileen (2023),Drama|Mystery
...,...,...
87576,Flaming Cloud (2023),(no genres listed)
87577,All Fun and Games (2023),Horror|Thriller
87581,Shelter in Solitude (2023),Comedy|Drama
87582,Orca (2023),Drama


In [27]:

# Count the number of ratings received by each movie
movie_rating_counts = (
    ratings.groupby("movieId")
    .size()
    .reset_index(name="number_of_ratings")
)

# Add movie titles and genres
popular_movies = movies.merge(
    movie_rating_counts,
    on="movieId",
    how="left"
)

# Sort by rating count
popular_movies = popular_movies.sort_values(
    "number_of_ratings",
    ascending=False
)

# Display the top 10 popular movies
display(
    popular_movies[
        ["title", "genres", "number_of_ratings"]
    ].head(10)
)


,title,genres,number_of_ratings
314,"Shawshank Redemption, The (1994)",Crime|Drama,102929.0
351,Forrest Gump (1994),Comedy|Drama|Romance|War,100296.0
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409.0
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808.0
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330.0
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010.0
2867,Fight Club (1999),Action|Crime|Drama|Thriller,77332.0
475,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233.0
522,Schindler's List (1993),Drama|War,73849.0
4888,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,73122.0


In [28]:

# Calculate average rating and number of ratings per movie
movie_statistics = ratings.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    number_of_ratings=("rating", "count")
).reset_index()

# Add movie titles and genres
movie_statistics = movie_statistics.merge(
    movies,
    on="movieId",
    how="left"
)

# Consider movies with at least 500 ratings
reliable_movies = movie_statistics[
    movie_statistics["number_of_ratings"] >= 500
].copy()

# Sort by average rating
top_rated_movies = reliable_movies.sort_values(
    "average_rating",
    ascending=False
)

display(
    top_rated_movies[
        ["title", "genres", "average_rating", "number_of_ratings"]
    ].head(15)
)


,title,genres,average_rating,number_of_ratings
44190,Planet Earth II (2016),Documentary,4.446830,1956
39306,Planet Earth (2006),Documentary,4.444369,2948
44042,Band of Brothers (2001),Action|Drama|War,4.426539,2811
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929
44416,Cosmos,(no genres listed),4.330081,615
840,"Godfather, The (1972)",Crime|Drama,4.317030,66440
58698,Parasite (2019),Comedy|Drama,4.312254,11670
47975,Blue Planet II (2017),Documentary,4.300086,1163
56760,Twin Peaks (1989),Drama|Mystery,4.298684,1140
1173,12 Angry Men (1957),Drama,4.265311,21863


In [29]:

# Count ratings submitted by each user
user_rating_counts = ratings.groupby("userId").size()

# Statistical summary of user activity
display(user_rating_counts.describe())

# Find the 10 most active users
most_active_users = user_rating_counts.sort_values(
    ascending=False
).head(10)

display(most_active_users)


count    200948.000000
mean        159.246193
std         282.025462
min          20.000000
25%          36.000000
50%          73.000000
75%         167.000000
max       33332.000000
dtype: float64

userId
175325    33332
17035      9577
55653      9178
123465     9044
171795     9016
10202      7748
198515     7594
49305      7488
22744      7372
7858       7322
dtype: int64

In [30]:

number_of_users = ratings["userId"].nunique()
number_of_movies = movies["movieId"].nunique()
actual_ratings = len(ratings)

total_possible_ratings = number_of_users * number_of_movies

sparsity = (
    1 - actual_ratings / total_possible_ratings
) * 100

print("Unique users:", number_of_users)
print("Unique movies:", number_of_movies)
print("Actual ratings:", actual_ratings)
print("Possible ratings:", total_possible_ratings)
print(f"Dataset sparsity: {sparsity:.2f}%")


Unique users: 200948
Unique movies: 87585
Actual ratings: 32000204
Possible ratings: 17600030580
Dataset sparsity: 99.82%


In [31]:
# Number of unique tags
print("Total tag records:", len(tags))
print("Unique tags:", tags["tag"].nunique())

# Find the most frequently used tags
popular_tags = tags["tag"].value_counts().head(20)

print("\nTop 20 most frequently used tags:")
display(popular_tags)


Total tag records: 2000072
Unique tags: 140979

Top 20 most frequently used tags:


tag
sci-fi                10996
atmospheric            9589
action                 8473
comedy                 8139
funny                  7467
surreal                7231
visually appealing     7090
based on a book        6617
twist ending           6521
dark comedy            6053
thought-provoking      6000
dystopia               5646
violence               5389
cinematography         5277
romance                5271
murder                 5180
social commentary      5152
stylized               4917
fantasy                4844
psychology             4827
Name: count, dtype: int64

In [32]:

# Select some familiar movies
sample_movie_titles = [
    "Toy Story (1995)",
    "The Matrix (1999)",
    "Pulp Fiction (1994)"
]

# Find their movie IDs
sample_movies = movies[
    movies["title"].isin(sample_movie_titles)
][["movieId", "title"]]

# Retrieve tags associated with these movies
sample_movie_tags = tags.merge(
    sample_movies,
    on="movieId",
    how="inner"
)

display(
    sample_movie_tags[
        ["title", "tag"]
    ].sort_values("title")
)


,title,tag
3963,Pulp Fiction (1994),gun pointed at face
5131,Pulp Fiction (1994),multiple storylines
5130,Pulp Fiction (1994),great dialogue
5129,Pulp Fiction (1994),good dialogue
5128,Pulp Fiction (1994),gangsters
...,...,...
5582,Toy Story (1995),Tom Hanks
5581,Toy Story (1995),Pixar
5580,Toy Story (1995),funny
1929,Toy Story (1995),adventure



## 4. Content-Based Filtering

Content-based filtering recommends movies by comparing their attributes. In this project, movie genres and user-generated tags are used to represent movie characteristics.

### 4.1 Tag Preprocessing

Missing tags are removed, and the remaining tags are converted to lowercase and cleaned. Tags belonging to the same movie are combined into a single text representation.

### 4.2 Movie Feature Construction

Movie genres and combined tags are merged to create a textual feature representation for each movie.

### 4.3 TF-IDF Feature Extraction

TF-IDF (Term Frequency–Inverse Document Frequency) converts the textual movie features into numerical vectors.

The vectorizer uses unigrams and bigrams to capture individual terms as well as pairs of consecutive terms.

### 4.4 Similarity-Based Recommendations

Cosine distance is used with the K-Nearest Neighbors algorithm to identify movies with similar feature representations.

Movies with smaller cosine distances are considered more similar.


In [33]:

# Create a separate copy for tag preprocessing
clean_tags = tags.dropna(subset=["tag"]).copy()

# Convert tags to lowercase and remove extra spaces
clean_tags["tag"] = (
    clean_tags["tag"]
    .str.lower()
    .str.strip()
)

# Remove empty tags
clean_tags = clean_tags[
    clean_tags["tag"] != ""
]

# Remove duplicate movie-tag combinations
clean_tags = clean_tags.drop_duplicates(
    subset=["movieId", "tag"]
)

print("Original tag records:", len(tags))
print("Cleaned tag records:", len(clean_tags))
print("Unique cleaned tags:", clean_tags["tag"].nunique())

display(clean_tags.head(10))


Original tag records: 2000072
Cleaned tag records: 1071041
Unique cleaned tags: 131644


,userId,movieId,tag,timestamp
0,22,26479,kevin kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102
5,34,8623,steve martin,1249808497
6,55,5766,the killls and the score,1319322078
7,58,7451,bullying,1672551536
8,58,7451,clique,1672551510
9,58,7451,coming of age,1672551502


In [34]:

# Combine all tags belonging to each movie
movie_tag_features = (
    clean_tags.groupby("movieId")["tag"]
    .apply(lambda movie_tags: " ".join(movie_tags))
    .reset_index(name="tag_text")
)

display(movie_tag_features.head(10))

print("Movies with tags:", len(movie_tag_features))


,movieId,tag_text
0,1,children disney animation pixar funny tumey's ...
1,2,robin williams fantasy time travel animals chi...
2,3,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,fantasy pregnancy remake family steve martin a...
5,6,atmospheric dialogue tumey's dvds realistic ac...
6,7,as good maybe better than original great cast ...
7,8,library system based on a book mark twain 19th...
8,9,action jean-claude van damme can't remember pe...
9,10,james bond tumey's dvds 007 bond casual sex es...


Movies with tags: 51323


In [35]:

# Prepare movie genres
movie_features = movies[["movieId", "title", "genres"]].copy()

# Replace genre separators with spaces
movie_features["genre_text"] = (
    movie_features["genres"]
    .str.replace("|", " ", regex=False)
)

# Add the combined tag information
movie_features = movie_features.merge(
    movie_tag_features,
    on="movieId",
    how="left"
)

# Replace missing tag descriptions with empty strings
movie_features["tag_text"] = (
    movie_features["tag_text"].fillna("")
)

# Combine genres and tags
movie_features["combined_features"] = (
    movie_features["genre_text"] + " " +
    movie_features["tag_text"]
).str.strip()

display(
    movie_features[
        ["title", "genre_text", "tag_text", "combined_features"]
    ].head(10)
)


,title,genre_text,tag_text,combined_features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy,children disney animation pixar funny tumey's ...,Adventure Animation Children Comedy Fantasy ch...
1,Jumanji (1995),Adventure Children Fantasy,robin williams fantasy time travel animals chi...,Adventure Children Fantasy robin williams fant...
2,Grumpier Old Men (1995),Comedy Romance,comedinha de velhinhos engraãƒâ§ada comedinha ...,Comedy Romance comedinha de velhinhos engraãƒâ...
3,Waiting to Exhale (1995),Comedy Drama Romance,characters slurs based on novel or book chick ...,Comedy Drama Romance characters slurs based on...
4,Father of the Bride Part II (1995),Comedy,fantasy pregnancy remake family steve martin a...,Comedy fantasy pregnancy remake family steve m...
5,Heat (1995),Action Crime Thriller,atmospheric dialogue tumey's dvds realistic ac...,Action Crime Thriller atmospheric dialogue tum...
6,Sabrina (1995),Comedy Romance,as good maybe better than original great cast ...,Comedy Romance as good maybe better than origi...
7,Tom and Huck (1995),Adventure Children,library system based on a book mark twain 19th...,Adventure Children library system based on a b...
8,Sudden Death (1995),Action,action jean-claude van damme can't remember pe...,Action action jean-claude van damme can't reme...
9,GoldenEye (1995),Action Adventure Thriller,james bond tumey's dvds 007 bond casual sex es...,Action Adventure Thriller james bond tumey's d...


In [36]:

print("Total movies:", len(movie_features))

print(
    "Movies with tags:",
    (movie_features["tag_text"] != "").sum()
)

print(
    "Movies without tags:",
    (movie_features["tag_text"] == "").sum()
)

print(
    "Missing combined features:",
    movie_features["combined_features"].isna().sum()
)


Total movies: 87585
Movies with tags: 51323
Movies without tags: 36262
Missing combined features: 0


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=100000
)

# Convert movie descriptions into TF-IDF vectors
tfidf_matrix = tfidf_vectorizer.fit_transform(
    movie_features["combined_features"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of non-zero values:", tfidf_matrix.nnz)

# Calculate memory used by the sparse matrix
matrix_memory = (
    tfidf_matrix.data.nbytes
    + tfidf_matrix.indices.nbytes
    + tfidf_matrix.indptr.nbytes
)

print(f"TF-IDF matrix memory: {matrix_memory / (1024 ** 2):.2f} MB")


TF-IDF matrix shape: (87585, 100000)
Number of non-zero values: 2465817
TF-IDF matrix memory: 28.55 MB


In [41]:

from sklearn.neighbors import NearestNeighbors

# Create the nearest-neighbor model
content_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

# Train the model using movie TF-IDF vectors
content_model.fit(tfidf_matrix)

print("Content-based nearest-neighbor model trained!")


Content-based nearest-neighbor model trained!


In [42]:

# Select a movie
movie_title = "Toy Story (1995)"

# Find its position in the movie feature dataset
movie_index = movie_features.index[
    movie_features["title"] == movie_title
][0]

# Retrieve the 11 nearest movies
distances, indices = content_model.kneighbors(
    tfidf_matrix[movie_index],
    n_neighbors=11
)

# Remove the input movie and prepare recommendations
recommended_indices = indices.flatten()[1:]
recommended_distances = distances.flatten()[1:]

recommendations = movie_features.iloc[
    recommended_indices
][["title", "genres"]].copy()

recommendations["similarity_score"] = (
    1 - recommended_distances
)

display(recommendations.reset_index(drop=True))


,title,genres,similarity_score
0,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.490809
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.473935
2,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.443523
3,Toy Story That Time Forgot (2014),Animation|Children,0.315681
4,Toy Story Toons: Partysaurus Rex (2012),Animation|Children|Comedy,0.297946
5,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,0.296202
6,Lamp Life (2020),Animation|Children|Comedy,0.283264
7,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,0.272759
8,Toy Masters (2014),Documentary,0.244001
9,Small Soldiers (1998),Animation|Children|Fantasy|War,0.243471


In [43]:

test_movies = [
    "Matrix, The (1999)",
    "Pulp Fiction (1994)",
    "Forrest Gump (1994)",
    "Shawshank Redemption, The (1994)"
]

for movie_title in test_movies:

    matching_indices = movie_features.index[
        movie_features["title"] == movie_title
    ]

    if len(matching_indices) == 0:
        print(f"Movie not found: {movie_title}")
        continue

    movie_index = matching_indices[0]

    distances, indices = content_model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=6
    )

    print(f"\nRecommendations for: {movie_title}")

    for index, distance in zip(
        indices.flatten()[1:],
        distances.flatten()[1:]
    ):
        print(
            movie_features.iloc[index]["title"],
            f"Similarity: {1 - distance:.3f}"
        )



Recommendations for: Matrix, The (1999)
Matrix Reloaded, The (2003) Similarity: 0.435
Matrix Revolutions, The (2003) Similarity: 0.422
Animatrix, The (2003) Similarity: 0.316
Terminator 2: Judgment Day (1991) Similarity: 0.314
The Matrix Resurrections (2021) Similarity: 0.304

Recommendations for: Pulp Fiction (1994)
Reservoir Dogs (1992) Similarity: 0.349
Django Unchained (2012) Similarity: 0.283
Seven Psychopaths (2012) Similarity: 0.279
Jackie Brown (1997) Similarity: 0.279
The Hateful Eight (2015) Similarity: 0.269

Recommendations for: Forrest Gump (1994)
Schindler's List (1993) Similarity: 0.403
Apartment, The (1960) Similarity: 0.390
Terms of Endearment (1983) Similarity: 0.382
Amadeus (1984) Similarity: 0.376
One Flew Over the Cuckoo's Nest (1975) Similarity: 0.369

Recommendations for: Shawshank Redemption, The (1994)
Green Mile, The (1999) Similarity: 0.295
Forrest Gump (1994) Similarity: 0.283
Pulp Fiction (1994) Similarity: 0.236
Clockwork Orange, A (1971) Similarity: 0.22

In [ ]:

# Search for the four movies without requiring exact title matches
search_terms = ["Matrix", "Pulp Fiction", "Forrest Gump", "Shawshank"]

for term in search_terms:
    print(f"\nSearching for: {term}")

    matching_movies = movies[
        movies["title"].str.contains(term, case=False, na=False)
    ]

    display(matching_movies[["movieId", "title"]].head(10))



Searching for: Matrix


,movieId,title
2480,2571,"Matrix, The (1999)"
6248,6365,"Matrix Reloaded, The (2003)"
6810,6934,"Matrix Revolutions, The (2003)"
9286,27660,"Animatrix, The (2003)"
28908,132490,Return to Source: The Philosophy of The Matrix...
40032,157721,Armitage: Dual Matrix (2002)
46862,172255,The Matrix Revisited (2001)
50298,179489,The Living Matrix (2009)
51063,181103,Matrix of Evil (2003)
67243,216897,Sex and the Matrix (2000)



Searching for: Pulp Fiction


,movieId,title
292,296,Pulp Fiction (1994)



Searching for: Forrest Gump


,movieId,title
351,356,Forrest Gump (1994)



Searching for: Shawshank


,movieId,title
314,318,"Shawshank Redemption, The (1994)"



## 5. Collaborative Filtering

Collaborative filtering generates recommendations using patterns in user ratings rather than relying only on movie metadata.

This project implements item-based collaborative filtering, where movies are compared based on the users who rated them.

### 5.1 User and Movie Encoding

Label Encoding converts original user IDs and movie IDs into consecutive integer indices suitable for matrix construction.

### 5.2 User-Item Matrix

A sparse user-item matrix is constructed, where rows represent users, columns represent movies, and non-zero entries represent ratings.

### 5.3 Item-User Matrix

The user-item matrix is transposed to obtain an item-user matrix. Each row now represents a movie and its ratings across users.

### 5.4 Nearest Neighbors

Cosine distance and K-Nearest Neighbors are used to identify movies with similar rating patterns.

Recommendations are generated using the similarity between movies already rated by a user.


In [44]:

from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder

# Convert user IDs and movie IDs into consecutive indices
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

user_indices = user_encoder.fit_transform(ratings["userId"])
movie_indices = movie_encoder.fit_transform(ratings["movieId"])

# Create the sparse user-item matrix
user_item_matrix = csr_matrix(
    (
        ratings["rating"].values,
        (user_indices, movie_indices)
    ),
    shape=(
        len(user_encoder.classes_),
        len(movie_encoder.classes_)
    )
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Number of stored ratings:", user_item_matrix.nnz)
print("Matrix format:", user_item_matrix.getformat())

matrix_memory = (
    user_item_matrix.data.nbytes
    + user_item_matrix.indices.nbytes
    + user_item_matrix.indptr.nbytes
)

print(f"Matrix memory: {matrix_memory / (1024 ** 2):.2f} MB")


User-item matrix shape: (200948, 84432)
Number of stored ratings: 32000204
Matrix format: csr
Matrix memory: 366.98 MB


In [45]:
from sklearn.neighbors import NearestNeighbors

# Convert user-item matrix into an item-user matrix
item_user_matrix = user_item_matrix.T.tocsr()

print("Item-user matrix shape:", item_user_matrix.shape)

# Create the collaborative filtering model
collaborative_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

# Fit the model on movie rating patterns
collaborative_model.fit(item_user_matrix)

print("Collaborative filtering model fitted successfully!")

Item-user matrix shape: (84432, 200948)
Collaborative filtering model fitted successfully!


In [46]:
# Select a movie to test
movie_title = "Matrix, The (1999)"

# Find its original MovieLens movie ID
movie_id = movies.loc[
    movies["title"] == movie_title, "movieId"
].iloc[0]

# Convert the original movie ID into its encoded index
movie_index = movie_encoder.transform([movie_id])[0]

# Find the 10 most similar movies
distances, indices = collaborative_model.kneighbors(
    item_user_matrix[movie_index],
    n_neighbors=11
)

# Remove the input movie itself
recommended_indices = indices.flatten()[1:]
recommended_distances = distances.flatten()[1:]

# Convert encoded indices back into original movie IDs
recommended_movie_ids = movie_encoder.inverse_transform(
    recommended_indices
)

# Prepare the recommendation results
recommendations = pd.DataFrame({
    "movieId": recommended_movie_ids,
    "similarity_score": 1 - recommended_distances
})

recommendations = recommendations.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

display(recommendations[["title", "genres", "similarity_score"]])

,title,genres,similarity_score
0,Fight Club (1999),Action|Crime|Drama|Thriller,0.693486
1,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.687291
2,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,0.666728
3,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.661939
4,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,0.658077
5,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.640040
6,Gladiator (2000),Action|Adventure|Drama,0.617098
7,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,0.616274
8,"Shawshank Redemption, The (1994)",Crime|Drama,0.612706
9,"Sixth Sense, The (1999)",Drama|Horror|Mystery,0.603012


In [47]:
test_movies = [
    "Toy Story (1995)",
    "Pulp Fiction (1994)",
    "Forrest Gump (1994)",
    "Shawshank Redemption, The (1994)"
]

for movie_title in test_movies:

    movie_id = movies.loc[
        movies["title"] == movie_title, "movieId"
    ].iloc[0]

    movie_index = movie_encoder.transform([movie_id])[0]

    distances, indices = collaborative_model.kneighbors(
        item_user_matrix[movie_index],
        n_neighbors=6
    )

    recommended_indices = indices.flatten()[1:]
    recommended_movie_ids = movie_encoder.inverse_transform(
        recommended_indices
    )

    recommended_titles = movies.set_index("movieId").loc[
        recommended_movie_ids, "title"
    ]

    print(f"\nRecommendations for {movie_title}:")
    for title in recommended_titles:
        print("-", title)


Recommendations for Toy Story (1995):
- Toy Story 2 (1999)
- Star Wars: Episode IV - A New Hope (1977)
- Back to the Future (1985)
- Forrest Gump (1994)
- Jurassic Park (1993)

Recommendations for Pulp Fiction (1994):
- Silence of the Lambs, The (1991)
- Shawshank Redemption, The (1994)
- Seven (a.k.a. Se7en) (1995)
- Usual Suspects, The (1995)
- Forrest Gump (1994)

Recommendations for Forrest Gump (1994):
- Shawshank Redemption, The (1994)
- Jurassic Park (1993)
- Silence of the Lambs, The (1991)
- Pulp Fiction (1994)
- Braveheart (1995)

Recommendations for Shawshank Redemption, The (1994):
- Pulp Fiction (1994)
- Forrest Gump (1994)
- Silence of the Lambs, The (1991)
- Schindler's List (1993)
- Usual Suspects, The (1995)



## 6. Preliminary Hybrid Recommendation Experiment

After developing the content-based and collaborative filtering approaches, an initial hybrid recommendation experiment is performed.

The experiment combines recommendation scores from both models after normalization.

The purpose of this stage is to explore how combining different recommendation signals can improve the recommendation results.

This preliminary experiment is followed by a separate temporal evaluation to assess model performance using a more structured train-test split.


In [48]:
def normalize_scores(scores):
    """Scale similarity scores between 0 and 1."""
    if not scores:
        return {}

    minimum = min(scores.values())
    maximum = max(scores.values())

    if maximum == minimum:
        return {movie_id: 1.0 for movie_id in scores}

    return {
        movie_id: (score - minimum) / (maximum - minimum)
        for movie_id, score in scores.items()
    }


def get_hybrid_recommendations(movie_title, top_n=10, alpha=0.5):

    # Find the movie in the metadata
    matching_movies = movie_features[
        movie_features["title"] == movie_title
    ]

    if matching_movies.empty:
        print("Movie not found!")
        return

    movie_id = matching_movies.iloc[0]["movieId"]
    content_index = matching_movies.index[0]

    content_scores = {}
    collaborative_scores = {}

    # 1. Get content-based recommendations
    distances, indices = content_model.kneighbors(
        tfidf_matrix[content_index],
        n_neighbors=31
    )

    for index, distance in zip(indices.flatten(), distances.flatten()):

        recommended_id = movie_features.iloc[index]["movieId"]

        if recommended_id != movie_id:
            content_scores[recommended_id] = 1 - distance

    # 2. Get collaborative recommendations
    collaborative_index = movie_encoder.transform([movie_id])[0]

    distances, indices = collaborative_model.kneighbors(
        item_user_matrix[collaborative_index],
        n_neighbors=31
    )

    recommended_ids = movie_encoder.inverse_transform(
        indices.flatten()
    )

    for recommended_id, distance in zip(
        recommended_ids, distances.flatten()
    ):

        if recommended_id != movie_id:
            collaborative_scores[recommended_id] = 1 - distance

    # 3. Normalize scores from both models
    content_scores = normalize_scores(content_scores)
    collaborative_scores = normalize_scores(collaborative_scores)

    # 4. Combine recommendations
    candidate_ids = set(content_scores) | set(collaborative_scores)

    hybrid_results = []

    for candidate_id in candidate_ids:

        content_score = content_scores.get(candidate_id, 0)
        collaborative_score = collaborative_scores.get(candidate_id, 0)

        hybrid_score = (
            alpha * content_score
            + (1 - alpha) * collaborative_score
        )

        hybrid_results.append({
            "movieId": candidate_id,
            "content_score": content_score,
            "collaborative_score": collaborative_score,
            "hybrid_score": hybrid_score
        })

    # 5. Sort and return the top recommendations
    results = pd.DataFrame(hybrid_results)

    results = results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    results = results.sort_values(
        "hybrid_score",
        ascending=False
    )

    return results[
        ["title", "genres", "content_score",
         "collaborative_score", "hybrid_score"]
    ].head(top_n).reset_index(drop=True)

In [49]:
hybrid_recommendations = get_hybrid_recommendations(
    movie_title="Matrix, The (1999)",
    top_n=10,
    alpha=0.5
)

display(hybrid_recommendations)

,title,genres,content_score,collaborative_score,hybrid_score
0,Fight Club (1999),Action|Crime|Drama|Thriller,0.000000,1.000000,0.500000
1,"Matrix Reloaded, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,1.000000,0.000000,0.500000
2,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.000000,0.958111,0.479056
3,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.312365,0.638630,0.475497
4,"Matrix Revolutions, The (2003)",Action|Adventure|Sci-Fi|Thriller|IMAX,0.937802,0.000000,0.468901
5,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,0.000000,0.819078,0.409539
6,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.000000,0.786697,0.393349
7,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,0.000000,0.760587,0.380293
8,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.424983,0.113688,0.269336
9,Gladiator (2000),Action|Adventure|Drama,0.000000,0.483515,0.241757


In [50]:
test_movies = [
    "Toy Story (1995)",
    "Pulp Fiction (1994)",
    "Forrest Gump (1994)",
    "Shawshank Redemption, The (1994)"
]

for movie_title in test_movies:

    print(f"HYBRID RECOMMENDATIONS FOR: {movie_title}")

    recommendations = get_hybrid_recommendations(
        movie_title=movie_title,
        top_n=5,
        alpha=0.5
    )

    display(recommendations)

HYBRID RECOMMENDATIONS FOR: Toy Story (1995)


,title,genres,content_score,collaborative_score,hybrid_score
0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.947913,1.000000,0.973956
1,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,1.000000,0.000000,0.500000
2,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.000000,0.877936,0.438968
3,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.854032,0.000000,0.427016
4,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.000000,0.727205,0.363602


HYBRID RECOMMENDATIONS FOR: Pulp Fiction (1994)


,title,genres,content_score,collaborative_score,hybrid_score
0,Reservoir Dogs (1992),Crime|Mystery|Thriller,1.000000,0.566320,0.783160
1,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,0.129881,1.000000,0.564941
2,"Shawshank Redemption, The (1994)",Crime|Drama,0.085301,0.872335,0.478818
3,Forrest Gump (1994),Comedy|Drama|Romance|War,0.208225,0.678278,0.443251
4,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,0.000000,0.757755,0.378878


HYBRID RECOMMENDATIONS FOR: Forrest Gump (1994)


,title,genres,content_score,collaborative_score,hybrid_score
0,Schindler's List (1993),Drama|War,1.000000,0.533436,0.766718
1,"Shawshank Redemption, The (1994)",Crime|Drama,0.000000,1.000000,0.500000
2,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.000000,0.932824,0.466412
3,"Apartment, The (1960)",Comedy|Drama|Romance,0.874907,0.000000,0.437454
4,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,0.000000,0.845343,0.422672


HYBRID RECOMMENDATIONS FOR: Shawshank Redemption, The (1994)


,title,genres,content_score,collaborative_score,hybrid_score
0,Forrest Gump (1994),Comedy|Drama|Romance|War,0.880721,0.933266,0.906993
1,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,0.437803,1.000000,0.718902
2,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,0.226910,0.909420,0.568165
3,"Green Mile, The (1999)",Crime|Drama,1.000000,0.000000,0.500000
4,Schindler's List (1993),Drama|War,0.000000,0.759753,0.379876



## 7. Temporal Train-Test Split

To evaluate the recommendation models, the ratings are divided into training and testing datasets based on their timestamps.

Instead of randomly splitting the ratings, a temporal cutoff is used:

- Ratings recorded on or before the cutoff are assigned to the training set.
- Ratings recorded after the cutoff are assigned to the testing set.

This approach simulates a real-world recommendation scenario in which historical interactions are used to predict future preferences.

### 7.1 Warm-Start Evaluation

The evaluation focuses on warm-start interactions, where both the user and the movie are already present in the training data.

This avoids evaluating recommendations for completely unseen users or movies, which would require separate cold-start handling.


In [51]:
# Inspect the rating timestamp range
print("Earliest rating:")
print(pd.to_datetime(ratings["timestamp"].min(), unit="s"))

print("\nLatest rating:")
print(pd.to_datetime(ratings["timestamp"].max(), unit="s"))

# Find the 80th percentile timestamp
split_timestamp = ratings["timestamp"].quantile(0.80)

print("\nTraining cutoff:")
print(pd.to_datetime(split_timestamp, unit="s"))

Earliest rating:
1995-01-09 11:46:44

Latest rating:
2023-10-13 02:29:07

Training cutoff:
2018-10-03 07:21:42


In [52]:
# Split ratings using the temporal cutoff
train_ratings = ratings[
    ratings["timestamp"] <= split_timestamp
].copy()

test_ratings = ratings[
    ratings["timestamp"] > split_timestamp
].copy()

print("Training ratings:", len(train_ratings))
print("Testing ratings:", len(test_ratings))

print(
    f"Training percentage: "
    f"{len(train_ratings) / len(ratings) * 100:.2f}%"
)

print(
    f"Testing percentage: "
    f"{len(test_ratings) / len(ratings) * 100:.2f}%"
)

Training ratings: 25600163
Testing ratings: 6400041
Training percentage: 80.00%
Testing percentage: 20.00%


In [53]:
train_users = set(train_ratings["userId"].unique())
test_users = set(test_ratings["userId"].unique())

train_movies = set(train_ratings["movieId"].unique())
test_movies = set(test_ratings["movieId"].unique())

unseen_test_users = test_users - train_users
unseen_test_movies = test_movies - train_movies

print("Users in test:", len(test_users))
print("Unseen test users:", len(unseen_test_users))

print("\nMovies in test:", len(test_movies))
print("Unseen test movies:", len(unseen_test_movies))

Users in test: 38272
Unseen test users: 30457

Movies in test: 77273
Unseen test movies: 33455


In [54]:
# Keep only test interactions involving known users and movies
warm_test_ratings = test_ratings[
    test_ratings["userId"].isin(train_users)
    & test_ratings["movieId"].isin(train_movies)
].copy()

print("Original test ratings:", len(test_ratings))
print("Warm-start test ratings:", len(warm_test_ratings))

print(
    "Retained percentage:",
    round(len(warm_test_ratings) / len(test_ratings) * 100, 2),
    "%"
)

print(
    "Excluded test ratings:",
    len(test_ratings) - len(warm_test_ratings)
)

Original test ratings: 6400041
Warm-start test ratings: 793859
Retained percentage: 12.4 %
Excluded test ratings: 5606182


In [55]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors

# Create encoders using training data only
train_user_encoder = LabelEncoder()
train_movie_encoder = LabelEncoder()

train_user_indices = train_user_encoder.fit_transform(
    train_ratings["userId"]
)

train_movie_indices = train_movie_encoder.fit_transform(
    train_ratings["movieId"]
)

# Build the training user-item matrix
train_user_item_matrix = csr_matrix(
    (
        train_ratings["rating"].values,
        (train_user_indices, train_movie_indices)
    ),
    shape=(
        len(train_user_encoder.classes_),
        len(train_movie_encoder.classes_)
    )
)

# Transpose to obtain the item-user matrix
train_item_user_matrix = train_user_item_matrix.T.tocsr()

print("Training user-item matrix:", train_user_item_matrix.shape)
print("Training item-user matrix:", train_item_user_matrix.shape)
print("Training ratings stored:", train_user_item_matrix.nnz)

# Calculate sparse matrix memory
matrix_memory = (
    train_user_item_matrix.data.nbytes
    + train_user_item_matrix.indices.nbytes
    + train_user_item_matrix.indptr.nbytes
)

print(f"Training matrix memory: {matrix_memory / (1024 ** 2):.2f} MB")

Training user-item matrix: (170491, 50977)
Training item-user matrix: (50977, 170491)
Training ratings stored: 25600163
Training matrix memory: 293.62 MB


In [56]:
train_collaborative_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

train_collaborative_model.fit(train_item_user_matrix)

print("Training-only collaborative model fitted!")

Training-only collaborative model fitted!



## 8. Evaluation Methodology

The recommendation models are evaluated using held-out test interactions.

A rating of 4.0 or higher is treated as a relevant interaction.

The evaluation uses a sample of 100 users to measure recommendation performance.

### 8.1 Evaluation Metrics

The following ranking metrics are used:

- **Precision@10:** Fraction of the top 10 recommended movies that are relevant.
- **Recall@10:** Fraction of relevant movies successfully included in the top 10 recommendations.
- **NDCG@10:** Measures ranking quality by assigning greater importance to relevant movies appearing near the top of the recommendation list.

Higher values indicate better recommendation performance.


In [57]:
# Identify relevant test interactions
relevant_test_ratings = warm_test_ratings[
    warm_test_ratings["rating"] >= 4.0
].copy()

# Count training and relevant test ratings per user
user_train_counts = train_ratings.groupby("userId").size()
user_test_counts = relevant_test_ratings.groupby("userId").size()

# Keep users with at least 5 training ratings
# and at least 1 relevant test rating
eligible_users = user_train_counts[
    user_train_counts >= 5
].index.intersection(user_test_counts.index)

print("Eligible evaluation users:", len(eligible_users))

# Select a reproducible sample of 100 users
evaluation_users = pd.Series(eligible_users).sample(
    n=min(100, len(eligible_users)),
    random_state=42
).tolist()

print("Selected evaluation users:", len(evaluation_users))
print("Relevant test interactions:", len(relevant_test_ratings))

Eligible evaluation users: 7438
Selected evaluation users: 100
Relevant test interactions: 312388


In [58]:
def recommend_for_user(user_id, top_n=10, seed_count=5, neighbors_per_movie=21):

    # Get this user's training history
    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    # Select their highest-rated movies
    favorite_movies = (
        user_history.sort_values("rating", ascending=False)
        .head(seed_count)
    )

    seen_movies = set(user_history["movieId"])

    recommendation_scores = {}

    # Find similar movies for each favorite movie
    for _, row in favorite_movies.iterrows():

        movie_id = row["movieId"]
        user_rating = row["rating"]

        movie_index = train_movie_encoder.transform([movie_id])[0]

        distances, indices = train_collaborative_model.kneighbors(
            train_item_user_matrix[movie_index],
            n_neighbors=neighbors_per_movie
        )

        for neighbor_index, distance in zip(
            indices.flatten(), distances.flatten()
        ):

            neighbor_id = train_movie_encoder.inverse_transform(
                [neighbor_index]
            )[0]

            # Skip movies the user has already rated
            if neighbor_id in seen_movies:
                continue

            similarity = max(0, 1 - distance)

            # Weight similarity by the user's rating
            score = similarity * (user_rating / 5.0)

            recommendation_scores[neighbor_id] = (
                recommendation_scores.get(neighbor_id, 0) + score
            )

    # Select the highest-scoring recommendations
    top_recommendations = sorted(
        recommendation_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_n]

    results = pd.DataFrame(
        top_recommendations,
        columns=["movieId", "recommendation_score"]
    )

    results = results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    return results[["title", "genres", "recommendation_score"]]


In [59]:
sample_user = evaluation_users[0]

print("Evaluation user ID:", sample_user)

print("\nRecommended movies:")
display(recommend_for_user(sample_user, top_n=10))

print("\nRelevant movies in the test set:")

user_relevant_movies = relevant_test_ratings[
    relevant_test_ratings["userId"] == sample_user
][["movieId", "rating"]].merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left"
)

display(
    user_relevant_movies.sort_values(
        "rating", ascending=False
    ).head(15)
)

Evaluation user ID: 20759

Recommended movies:


,title,genres,recommendation_score
0,Schindler's List (1993),Drama|War,1.748339
1,Braveheart (1995),Action|Drama|War,1.682417
2,Terminator 2: Judgment Day (1991),Action|Sci-Fi,1.630384
3,"Godfather, The (1972)",Crime|Drama,1.606472
4,"Fugitive, The (1993)",Thriller,1.602972
5,Memento (2000),Mystery|Thriller,1.368678
6,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,1.128728
7,Apollo 13 (1995),Adventure|Drama|IMAX,1.095529
8,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,1.095090
9,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.994535



Relevant movies in the test set:


,movieId,rating,title
3,858,5.0,"Godfather, The (1972)"
8,2329,5.0,American History X (1998)
1,628,4.5,Primal Fear (1996)
5,1221,4.5,"Godfather: Part II, The (1974)"
4,1203,4.5,12 Angry Men (1957)
7,2324,4.5,Life Is Beautiful (La Vita è bella) (1997)
0,527,4.5,Schindler's List (1993)
13,4326,4.5,Mississippi Burning (1988)
12,4226,4.5,Memento (2000)
10,3198,4.5,Papillon (1973)


In [60]:
import numpy as np

def calculate_metrics(recommended_ids, relevant_ids, k=10):

    recommended_ids = recommended_ids[:k]

    hits = [
        1 if movie_id in relevant_ids else 0
        for movie_id in recommended_ids
    ]

    # Precision@K
    precision = sum(hits) / k

    # Recall@K
    recall = (
        sum(hits) / len(relevant_ids)
        if relevant_ids else 0
    )

    # DCG@K
    dcg = sum(
        hit / np.log2(rank + 2)
        for rank, hit in enumerate(hits)
    )

    # Ideal DCG@K
    ideal_hits = min(len(relevant_ids), k)

    idcg = sum(
        1 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    # NDCG@K
    ndcg = dcg / idcg if idcg > 0 else 0

    return precision, recall, ndcg

In [61]:
evaluation_results = []

for user_id in evaluation_users:

    # Generate recommendations
    recommendations = recommend_for_user(
        user_id,
        top_n=10
    )

    # Get recommended movie IDs
    recommended_ids = movies[
        movies["title"].isin(recommendations["title"])
    ]["movieId"].tolist()

    # Get relevant future movies for this user
    relevant_ids = set(
        relevant_test_ratings.loc[
            relevant_test_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # Calculate metrics
    precision, recall, ndcg = calculate_metrics(
        recommended_ids,
        relevant_ids,
        k=10
    )

    evaluation_results.append({
        "userId": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })

# Convert results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df.head(10))

print("\nAverage Evaluation Metrics:")
display(
    evaluation_df[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().to_frame("Average Score")
)

,userId,Precision@10,Recall@10,NDCG@10
0,20759,0.3,0.096774,0.214084
1,177585,0.0,0.000000,0.000000
2,197165,0.5,0.010417,0.593361
3,199798,0.1,0.020408,0.220092
4,62370,0.2,0.027027,0.136985
5,13276,0.0,0.000000,0.000000
6,20843,0.1,0.025000,0.094788
7,61745,0.0,0.000000,0.000000
8,151510,0.4,0.030769,0.519997
9,47442,0.0,0.000000,0.000000



Average Evaluation Metrics:


,Average Score
Precision@10,0.101000
Recall@10,0.031327
NDCG@10,0.102631


In [62]:
def recommend_for_user(user_id, top_n=10, seed_count=5):

    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    favorite_movies = (
        user_history.sort_values("rating", ascending=False)
        .head(seed_count)
    )

    seen_movies = set(user_history["movieId"])
    recommendation_scores = {}

    for _, row in favorite_movies.iterrows():

        movie_id = row["movieId"]
        user_rating = row["rating"]

        movie_index = train_movie_encoder.transform([movie_id])[0]

        distances, indices = train_collaborative_model.kneighbors(
            train_item_user_matrix[movie_index],
            n_neighbors=21
        )

        for neighbor_index, distance in zip(
            indices.flatten(), distances.flatten()
        ):

            neighbor_id = train_movie_encoder.inverse_transform(
                [neighbor_index]
            )[0]

            if neighbor_id in seen_movies:
                continue

            similarity = max(0, 1 - distance)
            score = similarity * (user_rating / 5.0)

            recommendation_scores[neighbor_id] = (
                recommendation_scores.get(neighbor_id, 0) + score
            )

    top_recommendations = sorted(
        recommendation_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_n]

    results = pd.DataFrame(
        top_recommendations,
        columns=["movieId", "recommendation_score"]
    )

    results = results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    return results[
        ["movieId", "title", "genres", "recommendation_score"]
    ]

In [63]:
evaluation_results = []

for user_id in evaluation_users:

    recommendations = recommend_for_user(user_id, top_n=10)

    recommended_ids = recommendations["movieId"].tolist()

    relevant_ids = set(
        relevant_test_ratings.loc[
            relevant_test_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    precision, recall, ndcg = calculate_metrics(
        recommended_ids,
        relevant_ids,
        k=10
    )

    evaluation_results.append({
        "userId": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Average metrics:")
display(
    evaluation_df[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().to_frame("Average Score")
)

Average metrics:


,Average Score
Precision@10,0.101000
Recall@10,0.031327
NDCG@10,0.118156



## 9. Popularity-Based Baseline

A popularity-based recommendation approach is implemented as a baseline for comparison.

Movie popularity is calculated using training data only, based on the number of ratings and average rating received by each movie.

Movies already rated by a user are excluded from their recommendations.

The baseline provides a reference point for determining whether personalized recommendation models offer an improvement over recommending popular movies.


In [64]:
# Calculate movie popularity using training data only
movie_popularity = (
    train_ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

# Require at least 100 ratings to avoid unreliable averages
movie_popularity = movie_popularity[
    movie_popularity["rating_count"] >= 100
].copy()

# Calculate a Bayesian-style weighted rating
global_mean = train_ratings["rating"].mean()
minimum_votes = 100

movie_popularity["weighted_score"] = (
    (movie_popularity["rating_count"] /
     (movie_popularity["rating_count"] + minimum_votes))
    * movie_popularity["average_rating"]
    +
    (minimum_votes /
     (movie_popularity["rating_count"] + minimum_votes))
    * global_mean
)

# Sort by weighted score
movie_popularity = movie_popularity.sort_values(
    "weighted_score",
    ascending=False
)

# Display the most popular movies
display(
    movie_popularity.merge(
        movies[["movieId", "title"]],
        on="movieId"
    )[["title", "rating_count", "average_rating", "weighted_score"]]
    .head(10)
)

,title,rating_count,average_rating,weighted_score
0,"Shawshank Redemption, The (1994)",84098,4.416746,4.415695
1,Planet Earth (2006),1310,4.469847,4.403324
2,Planet Earth II (2016),770,4.494805,4.384123
3,"Godfather, The (1972)",53953,4.329416,4.327941
4,Band of Brothers (2001),931,4.407626,4.322684
5,"Usual Suspects, The (1995)",57807,4.292240,4.290927
6,"Godfather: Part II, The (1974)",35172,4.261458,4.259390
7,Schindler's List (1993),63076,4.254701,4.253557
8,Seven Samurai (Shichinin no samurai) (1954),13934,4.255131,4.249978
9,Rear Window (1954),20980,4.237369,4.234022


In [65]:
# Store globally ranked movie IDs
popular_movie_ids = movie_popularity["movieId"].tolist()

popularity_results = []

for user_id in evaluation_users:

    # Find movies already rated by this user
    seen_movies = set(
        train_ratings.loc[
            train_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # Recommend the most highly ranked unseen movies
    recommended_ids = [
        movie_id
        for movie_id in popular_movie_ids
        if movie_id not in seen_movies
    ][:10]

    # Find relevant future movies
    relevant_ids = set(
        relevant_test_ratings.loc[
            relevant_test_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    # Calculate evaluation metrics
    precision, recall, ndcg = calculate_metrics(
        recommended_ids,
        relevant_ids,
        k=10
    )

    popularity_results.append({
        "userId": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })

popularity_evaluation_df = pd.DataFrame(popularity_results)

print("Popularity-based evaluation:")
display(
    popularity_evaluation_df[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().to_frame("Average Score")
)

Popularity-based evaluation:


,Average Score
Precision@10,0.056000
Recall@10,0.023596
NDCG@10,0.063029



## 10. Content-Based Personalized Evaluation

The content-based model is extended to generate personalized recommendations using a user's rating history.

Movies rated highly by a user are used as reference movies.

Similar movies are retrieved using the content-based model, and movies already rated by the user are excluded.

The resulting recommendations are evaluated using Precision@10, Recall@10 and NDCG@10.


In [66]:
# Map original movie IDs to their content-feature row indices
content_movie_index = dict(
    zip(movie_features["movieId"], movie_features.index)
)

train_movie_ids = set(train_movie_encoder.classes_)


def recommend_content_for_user(user_id, top_n=10, seed_count=5):

    user_history = train_ratings[
        train_ratings["userId"] == user_id
    ]

    favorite_movies = (
        user_history.sort_values("rating", ascending=False)
        .head(seed_count)
    )

    seen_movies = set(user_history["movieId"])
    recommendation_scores = {}

    for _, row in favorite_movies.iterrows():

        movie_id = row["movieId"]
        user_rating = row["rating"]

        if movie_id not in content_movie_index:
            continue

        content_index = content_movie_index[movie_id]

        distances, indices = content_model.kneighbors(
            tfidf_matrix[content_index],
            n_neighbors=21
        )

        for neighbor_index, distance in zip(
            indices.flatten(), distances.flatten()
        ):

            neighbor_id = movie_features.iloc[neighbor_index]["movieId"]

            # Keep only unseen movies available in training
            if neighbor_id in seen_movies or neighbor_id not in train_movie_ids:
                continue

            similarity = max(0, 1 - distance)
            score = similarity * (user_rating / 5.0)

            recommendation_scores[neighbor_id] = (
                recommendation_scores.get(neighbor_id, 0) + score
            )

    top_recommendations = sorted(
        recommendation_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_n]

    results = pd.DataFrame(
        top_recommendations,
        columns=["movieId", "recommendation_score"]
    )

    return results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

In [67]:
sample_user = evaluation_users[0]

print("User ID:", sample_user)

display(
    recommend_content_for_user(sample_user, top_n=10)
)

User ID: 20759


,movieId,recommendation_score,title,genres
0,97306,0.480768,Seven Psychopaths (2012),Comedy|Crime
1,858,0.456717,"Godfather, The (1972)",Crime|Drama
2,6874,0.453519,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
3,1193,0.420810,One Flew Over the Cuckoo's Nest (1975),Drama
4,99114,0.282718,Django Unchained (2012),Action|Drama|Western
5,1729,0.278921,Jackie Brown (1997),Crime|Drama|Thriller
6,128360,0.268844,The Hateful Eight (2015),Western
7,6548,0.244725,Bad Boys II (2003),Action|Comedy|Crime|Thriller
8,88129,0.243777,Drive (2011),Crime|Drama|Film-Noir|Thriller
9,493,0.240190,Menace II Society (1993),Action|Crime|Drama


In [69]:
content_evaluation_results = []

for user_id in evaluation_users:

    recommendations = recommend_content_for_user(
        user_id,
        top_n=10
    )

    recommended_ids = recommendations["movieId"].tolist()

    relevant_ids = set(
        relevant_test_ratings.loc[
            relevant_test_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    precision, recall, ndcg = calculate_metrics(
        recommended_ids,
        relevant_ids,
        k=10
    )

    content_evaluation_results.append({
        "userId": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })

content_evaluation_df = pd.DataFrame(content_evaluation_results)

print("Content-based evaluation:")
display(
    content_evaluation_df[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().to_frame("Average Score")
)

Content-based evaluation:


,Average Score
Precision@10,0.051000
Recall@10,0.024020
NDCG@10,0.060749



## 11. Hybrid Recommendation System

The hybrid model combines content-based and collaborative recommendations to leverage the strengths of both approaches.

Content-based recommendations capture similarities in movie genres and tags, while collaborative recommendations capture patterns in user rating behaviour.

### 11.1 Candidate Recommendations

Recommendations are generated independently by both models.

The candidate lists are then combined while excluding movies already rated by the user.

### 11.2 Reciprocal Rank Fusion

Reciprocal Rank Fusion (RRF) combines recommendation lists using their ranking positions rather than directly comparing raw scores.

A movie receives a higher combined score when it appears near the top of the recommendation lists.


In [70]:
def recommend_hybrid_for_user(user_id, top_n=10, candidate_count=50):

    # Get recommendations from both models
    content_recommendations = recommend_content_for_user(
        user_id,
        top_n=candidate_count
    )

    collaborative_recommendations = recommend_for_user(
        user_id,
        top_n=candidate_count
    )

    # Store combined ranking scores
    hybrid_scores = {}

    # Add content-based ranking contributions
    for rank, movie_id in enumerate(
        content_recommendations["movieId"], start=1
    ):
        hybrid_scores[movie_id] = (
            hybrid_scores.get(movie_id, 0)
            + 1 / (60 + rank)
        )

    # Add collaborative ranking contributions
    for rank, movie_id in enumerate(
        collaborative_recommendations["movieId"], start=1
    ):
        hybrid_scores[movie_id] = (
            hybrid_scores.get(movie_id, 0)
            + 1 / (60 + rank)
        )

    # Sort by combined score
    top_recommendations = sorted(
        hybrid_scores.items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_n]

    results = pd.DataFrame(
        top_recommendations,
        columns=["movieId", "hybrid_score"]
    )

    return results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

In [71]:
sample_user = evaluation_users[0]

print("User ID:", sample_user)

hybrid_user_recommendations = recommend_hybrid_for_user(
    sample_user,
    top_n=10
)

display(hybrid_user_recommendations)

User ID: 20759


,movieId,hybrid_score,title,genres
0,858,0.031754,"Godfather, The (1972)",Crime|Drama
1,589,0.029958,Terminator 2: Judgment Day (1991),Action|Sci-Fi
2,6874,0.029572,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
3,4226,0.028850,Memento (2000),Mystery|Thriller
4,46723,0.024961,Babel (2006),Drama|Thriller
5,3578,0.022024,Gladiator (2000),Action|Adventure|Drama
6,6953,0.021904,21 Grams (2003),Crime|Drama|Mystery|Romance|Thriller
7,97306,0.016393,Seven Psychopaths (2012),Comedy|Crime
8,527,0.016393,Schindler's List (1993),Drama|War
9,110,0.016129,Braveheart (1995),Action|Drama|War


In [72]:
hybrid_evaluation_results = []

for user_id in evaluation_users:

    recommendations = recommend_hybrid_for_user(
        user_id,
        top_n=10
    )

    recommended_ids = recommendations["movieId"].tolist()

    relevant_ids = set(
        relevant_test_ratings.loc[
            relevant_test_ratings["userId"] == user_id,
            "movieId"
        ]
    )

    precision, recall, ndcg = calculate_metrics(
        recommended_ids,
        relevant_ids,
        k=10
    )

    hybrid_evaluation_results.append({
        "userId": user_id,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })

hybrid_evaluation_df = pd.DataFrame(
    hybrid_evaluation_results
)

print("Hybrid evaluation:")
display(
    hybrid_evaluation_df[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ].mean().to_frame("Average Score")
)

Hybrid evaluation:


,Average Score
Precision@10,0.095000
Recall@10,0.035886
NDCG@10,0.118923



## 12. Weighted Reciprocal Rank Fusion Experiments

To investigate the contribution of each recommendation approach, different weights are assigned to the content-based and collaborative recommendation lists.

The weighted RRF method combines both ranking signals using:

\[
RRF(m) =
\frac{w_c}{k + rank_c(m)}
+
\frac{w_{cf}}{k + rank_{cf}(m)}
\]

where:

- \(w_c\) is the content-based weight.
- \(w_{cf}\) is the collaborative filtering weight.
- \(rank_c(m)\) is the content-based rank.
- \(rank_{cf}(m)\) is the collaborative rank.
- \(k\) is the RRF constant.

Different weight configurations are evaluated using Precision@10, Recall@10 and NDCG@10.


In [73]:
cached_recommendations = {}

for user_id in evaluation_users:

    content_ids = recommend_content_for_user(
        user_id, top_n=50
    )["movieId"].tolist()

    collaborative_ids = recommend_for_user(
        user_id, top_n=50
    )["movieId"].tolist()

    cached_recommendations[user_id] = {
        "content": content_ids,
        "collaborative": collaborative_ids
    }

print("Cached recommendations for", len(cached_recommendations), "users.")

Cached recommendations for 100 users.


In [74]:
def weighted_rank_fusion(
    content_ids,
    collaborative_ids,
    content_weight=0.3,
    collaborative_weight=0.7,
    top_n=10,
    rrf_constant=60
):

    scores = {}

    for rank, movie_id in enumerate(content_ids, start=1):
        scores[movie_id] = (
            scores.get(movie_id, 0)
            + content_weight / (rrf_constant + rank)
        )

    for rank, movie_id in enumerate(collaborative_ids, start=1):
        scores[movie_id] = (
            scores.get(movie_id, 0)
            + collaborative_weight / (rrf_constant + rank)
        )

    ranked_movies = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True
    )

    return [movie_id for movie_id, score in ranked_movies[:top_n]]

In [75]:
weight_experiments = [
    ("Equal (50:50)", 0.5, 0.5),
    ("Collaborative-heavy (30:70)", 0.3, 0.7),
    ("Collaborative-dominant (20:80)", 0.2, 0.8)
]

weight_results = []

for experiment_name, content_weight, collaborative_weight in weight_experiments:

    experiment_metrics = []

    for user_id in evaluation_users:

        content_ids = cached_recommendations[user_id]["content"]
        collaborative_ids = cached_recommendations[user_id]["collaborative"]

        recommended_ids = weighted_rank_fusion(
            content_ids,
            collaborative_ids,
            content_weight=content_weight,
            collaborative_weight=collaborative_weight,
            top_n=10
        )

        relevant_ids = set(
            relevant_test_ratings.loc[
                relevant_test_ratings["userId"] == user_id,
                "movieId"
            ]
        )

        precision, recall, ndcg = calculate_metrics(
            recommended_ids,
            relevant_ids,
            k=10
        )

        experiment_metrics.append([precision, recall, ndcg])

    average_metrics = np.mean(experiment_metrics, axis=0)

    weight_results.append({
        "Experiment": experiment_name,
        "Precision@10": average_metrics[0],
        "Recall@10": average_metrics[1],
        "NDCG@10": average_metrics[2]
    })

weight_comparison = pd.DataFrame(weight_results)

display(weight_comparison)

,Experiment,Precision@10,Recall@10,NDCG@10
0,Equal (50:50),0.095,0.035886,0.118923
1,Collaborative-heavy (30:70),0.110,0.041759,0.131310
2,Collaborative-dominant (20:80),0.114,0.042281,0.133015


In [76]:

remaining_experiments = [
    ("Content-only (100:0)", 1.0, 0.0),
    ("Content-heavy (80:20)", 0.8, 0.2),
    ("Content-leaning (60:40)", 0.6, 0.4),
    ("Collaborative-leaning (40:60)", 0.4, 0.6),
    ("Collaborative-dominant (10:90)", 0.1, 0.9),
    ("Collaborative-only (0:100)", 0.0, 1.0)
]

additional_results = []

for experiment_name, content_weight, collaborative_weight in remaining_experiments:

    experiment_metrics = []

    for user_id in evaluation_users:

        content_ids = cached_recommendations[user_id]["content"]
        collaborative_ids = cached_recommendations[user_id]["collaborative"]

        recommended_ids = weighted_rank_fusion(
            content_ids,
            collaborative_ids,
            content_weight=content_weight,
            collaborative_weight=collaborative_weight,
            top_n=10
        )

        relevant_ids = set(
            relevant_test_ratings.loc[
                relevant_test_ratings["userId"] == user_id,
                "movieId"
            ]
        )

        precision, recall, ndcg = calculate_metrics(
            recommended_ids,
            relevant_ids,
            k=10
        )

        experiment_metrics.append([precision, recall, ndcg])

    average_metrics = np.mean(experiment_metrics, axis=0)

    additional_results.append({
        "Experiment": experiment_name,
        "Precision@10": average_metrics[0],
        "Recall@10": average_metrics[1],
        "NDCG@10": average_metrics[2]
    })

# Combine with the three experiments already completed
final_weight_results = pd.concat(
    [
        weight_comparison,
        pd.DataFrame(additional_results)
    ],
    ignore_index=True
)

display(
    final_weight_results.sort_values(
        "NDCG@10",
        ascending=False
    ).reset_index(drop=True)
)


,Experiment,Precision@10,Recall@10,NDCG@10
0,Collaborative-dominant (20:80),0.114,0.042281,0.133015
1,Collaborative-heavy (30:70),0.110,0.041759,0.131310
2,Collaborative-leaning (40:60),0.110,0.042644,0.130640
3,Collaborative-dominant (10:90),0.108,0.041031,0.126286
4,Equal (50:50),0.095,0.035886,0.118923
5,Collaborative-only (0:100),0.101,0.031327,0.118156
6,Content-leaning (60:40),0.075,0.032379,0.101145
7,Content-heavy (80:20),0.065,0.028029,0.087718
8,Content-only (100:0),0.051,0.024020,0.060749



## 13. Final Hybrid Model

Based on the weight experiments, the final configuration assigns:

- Content-Based Filtering: 20%
- Collaborative Filtering: 80%

This configuration was selected based on its performance in the evaluated sample, particularly its Precision@10 and NDCG@10 results.

The final recommendation function combines both recommendation lists using weighted Reciprocal Rank Fusion.


In [77]:

FINAL_CONTENT_WEIGHT = 0.2
FINAL_COLLABORATIVE_WEIGHT = 0.8

def recommend_final_hybrid(user_id, top_n=10):
    content_ids = cached_recommendations[user_id]["content"]
    collaborative_ids = cached_recommendations[user_id]["collaborative"]

    recommended_ids = weighted_rank_fusion(
        content_ids,
        collaborative_ids,
        content_weight=FINAL_CONTENT_WEIGHT,
        collaborative_weight=FINAL_COLLABORATIVE_WEIGHT,
        top_n=top_n
    )

    results = pd.DataFrame({"movieId": recommended_ids})

    return results.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

print("Final hybrid configuration: 20% Content + 80% Collaborative")
display(recommend_final_hybrid(20759))


Final hybrid configuration: 20% Content + 80% Collaborative


,movieId,title,genres
0,858,"Godfather, The (1972)",Crime|Drama
1,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
2,4226,Memento (2000),Mystery|Thriller
3,6874,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
4,527,Schindler's List (1993),Drama|War
5,110,Braveheart (1995),Action|Drama|War
6,457,"Fugitive, The (1993)",Thriller
7,46723,Babel (2006),Drama|Thriller
8,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
9,3578,Gladiator (2000),Action|Adventure|Drama



## 14. Saving Model Artifacts

After completing model development and evaluation, the trained models and supporting data structures are saved for reuse.

The following artifacts are stored:

1. **content_artifacts.joblib:** Contains movie metadata, movie features, the TF-IDF matrix and the trained content-based nearest-neighbor model.
2. **collaborative_artifacts.joblib:** Contains user and movie encoders, sparse rating matrices and the trained collaborative nearest-neighbor model.
3. **train_ratings.parquet:** Contains the training ratings used by the recommendation application.

These artifacts allow the Streamlit application to load the trained models without repeating the complete training process.


In [1]:
import joblib

print("Loading content artifacts...")
content = joblib.load("artifacts/content_artifacts.joblib")
print("Content artifacts loaded!")

print("Loading collaborative artifacts...")
collaborative = joblib.load("artifacts/collaborative_artifacts.joblib")
print("Collaborative artifacts loaded!")

Loading content artifacts...
Content artifacts loaded!
Loading collaborative artifacts...
Collaborative artifacts loaded!
